# Gradient Descent Optimization

Gradient descent is a fundamental optimization algorithm used in machine learning to minimize the cost/loss function by iteratively moving toward the steepest descent as defined by the negative of the gradient. This notebook explores gradient descent from theoretical foundations to practical implementations.

## Table of Contents
1. [Import Required Libraries](#import-required-libraries)
2. [Understanding Gradient Descent](#understanding-gradient-descent)
3. [Mathematical Foundations](#mathematical-foundations)
4. [Implementing Batch Gradient Descent](#implementing-batch-gradient-descent)
5. [Stochastic Gradient Descent (SGD)](#stochastic-gradient-descent-sgd)
6. [Mini-Batch Gradient Descent](#mini-batch-gradient-descent)
7. [Gradient Descent Variants](#gradient-descent-variants)
8. [Challenges and Solutions](#challenges-and-solutions)
9. [Visualization of Convergence](#visualization-of-convergence)
10. [Practical Examples](#practical-examples)

## Import Required Libraries

In [ ]:
# Import core libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import animation, cm
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import HTML

# Machine learning libraries
import sklearn
from sklearn.datasets import make_regression, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, log_loss

# Deep learning libraries (if available)
try:
    import tensorflow as tf
    from tensorflow import keras
    tf_available = True
except ImportError:
    print("TensorFlow not available. Some deep learning examples will be skipped.")
    tf_available = False

# Set random seed for reproducibility
np.random.seed(42)
if tf_available:
    tf.random.set_seed(42)
    
# Configure matplotlib for better visualizations
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12

print("Libraries imported successfully!")

## Understanding Gradient Descent

Gradient descent is an iterative optimization algorithm used to find the minimum of a function. In machine learning, we use gradient descent to minimize the cost/loss function of our models.

### Key Concepts:

1. **Cost/Loss Function**: A function that measures how far our predictions are from the actual values
2. **Gradient**: The vector of partial derivatives, pointing in the direction of steepest increase
3. **Learning Rate**: A hyperparameter that determines the size of steps we take during optimization
4. **Convergence**: The process of reaching the minimum of the cost function

### How Gradient Descent Works:

1. Start with initial parameter values
2. Calculate the gradient of the cost function with respect to each parameter
3. Update parameters by moving in the opposite direction of the gradient
4. Repeat until convergence

The parameter update rule is:
$\theta_{new} = \theta_{old} - \alpha \nabla J(\theta)$

Where:
- $\theta$ represents the parameters
- $\alpha$ is the learning rate
- $\nabla J(\theta)$ is the gradient of the cost function with respect to the parameters

## Mathematical Foundations

Let's explore the mathematical foundations of gradient descent by considering a simple function and calculating its gradient.

### Example: Simple Quadratic Function

Consider the function $f(x) = x^2$. The derivative of this function is $f'(x) = 2x$.

For a multivariable function $f(x, y) = x^2 + y^2$, the gradient is:
$\nabla f(x, y) = \begin{bmatrix} \frac{\partial f}{\partial x} \\ \frac{\partial f}{\partial y} \end{bmatrix} = \begin{bmatrix} 2x \\ 2y \end{bmatrix}$

Let's implement this and visualize:

In [ ]:
def simple_function(x):
    """A simple quadratic function f(x) = x^2"""
    return x**2

def gradient_simple_function(x):
    """Gradient of f(x) = x^2, which is f'(x) = 2x"""
    return 2*x

# Visualize the function and its gradient
x_values = np.linspace(-5, 5, 100)
y_values = simple_function(x_values)
gradients = gradient_simple_function(x_values)

plt.figure(figsize=(12, 6))

# Plot the function
plt.subplot(1, 2, 1)
plt.plot(x_values, y_values)
plt.title("Function: f(x) = x²")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(True)

# Plot the gradient
plt.subplot(1, 2, 2)
plt.plot(x_values, gradients)
plt.title("Gradient: f'(x) = 2x")
plt.xlabel("x")
plt.ylabel("f'(x)")
plt.grid(True)

plt.tight_layout()
plt.show()

### 2D Function Example: Gradient Calculation

Let's consider a 2D function $f(x, y) = x^2 + y^2$ and calculate its gradient.

In [ ]:
def function_2d(x, y):
    """A simple 2D quadratic function f(x,y) = x^2 + y^2"""
    return x**2 + y**2

def gradient_2d(x, y):
    """Gradient of f(x,y) = x^2 + y^2, which is [2x, 2y]"""
    return np.array([2*x, 2*y])

# Create meshgrid for 3D plotting
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 100)
X, Y = np.meshgrid(x, y)
Z = function_2d(X, Y)

# Plot the 3D surface
fig = plt.figure(figsize=(14, 8))
ax = fig.add_subplot(1, 2, 1, projection='3d')
surface = ax.plot_surface(X, Y, Z, cmap=cm.viridis, alpha=0.8)
ax.set_title("Function: f(x,y) = x² + y²")
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_zlabel('f(x,y)')

# Plot contour and gradient field
ax2 = fig.add_subplot(1, 2, 2)
contour = ax2.contour(X, Y, Z, levels=20, cmap=cm.viridis)
ax2.set_title("Contour Plot with Gradient Vectors")

# Add gradient vectors at selected points
x_points = np.linspace(-4, 4, 9)
y_points = np.linspace(-4, 4, 9)
X_pts, Y_pts = np.meshgrid(x_points, y_points)
U = np.zeros_like(X_pts)
V = np.zeros_like(Y_pts)

for i in range(len(x_points)):
    for j in range(len(y_points)):
        grad = gradient_2d(X_pts[i, j], Y_pts[i, j])
        U[i, j] = -grad[0]  # Negative because we descend the gradient
        V[i, j] = -grad[1]  # Negative because we descend the gradient

ax2.quiver(X_pts, Y_pts, U, V, color='red', scale=100)
ax2.set_xlabel('x')
ax2.set_ylabel('y')

plt.colorbar(contour, ax=ax2)
plt.tight_layout()
plt.show()

## Implementing Batch Gradient Descent

In batch gradient descent, we use the entire dataset to compute the gradient of the cost function and update the parameters accordingly. This approach ensures a stable convergence but can be computationally expensive for large datasets.

Let's implement batch gradient descent for a simple linear regression problem:

In [ ]:
# Generate synthetic regression data
X, y = make_regression(n_samples=100, n_features=1, noise=10, random_state=42)

# Function to compute the cost (mean squared error)
def compute_cost(X, y, theta):
    """
    Compute the mean squared error cost function
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples,)
        theta: Model parameters (n_features + 1,) including bias
        
    Returns:
        cost: Mean squared error
    """
    m = len(y)
    X_b = np.c_[np.ones((m, 1)), X]  # Add intercept term
    predictions = X_b.dot(theta)
    return (1/(2*m)) * np.sum(np.square(predictions - y))

# Function to compute the gradient
def compute_gradient(X, y, theta):
    """
    Compute the gradient of the cost function
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples,)
        theta: Model parameters (n_features + 1,) including bias
        
    Returns:
        gradient: Gradient of the cost function with respect to theta
    """
    m = len(y)
    X_b = np.c_[np.ones((m, 1)), X]  # Add intercept term
    predictions = X_b.dot(theta)
    error = predictions - y
    return (1/m) * X_b.T.dot(error)

# Batch gradient descent implementation
def batch_gradient_descent(X, y, learning_rate=0.01, n_iterations=1000):
    """
    Implement batch gradient descent for linear regression
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples,)
        learning_rate: Learning rate alpha
        n_iterations: Number of iterations
        
    Returns:
        theta: Optimized model parameters
        cost_history: Cost at each iteration
    """
    m, n = X.shape
    theta = np.zeros(n + 1)  # Initialize parameters (including bias)
    cost_history = np.zeros(n_iterations)
    theta_history = np.zeros((n_iterations, n + 1))
    
    for i in range(n_iterations):
        gradient = compute_gradient(X, y, theta)
        theta = theta - learning_rate * gradient
        cost = compute_cost(X, y, theta)
        
        # Store history for visualization
        cost_history[i] = cost
        theta_history[i] = theta
        
    return theta, cost_history, theta_history

# Run batch gradient descent
learning_rate = 0.01
n_iterations = 1000
theta, cost_history, theta_history = batch_gradient_descent(X, y, learning_rate, n_iterations)

# Plot the cost history
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_iterations + 1), cost_history, 'b-')
plt.title('Cost vs. Iteration')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot the final regression line
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='b', label='Data Points')

# Sort X for clean line plotting
X_sorted = np.sort(X, axis=0)
X_b_sorted = np.c_[np.ones((len(X_sorted), 1)), X_sorted]
y_pred = X_b_sorted.dot(theta)

plt.plot(X_sorted, y_pred, 'r-', label=f'Linear Regression: y = {theta[0]:.2f} + {theta[1]:.2f}x')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Batch Gradient Descent: Linear Regression')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Stochastic Gradient Descent (SGD)

While batch gradient descent uses the entire training set to compute gradients, stochastic gradient descent (SGD) uses only a single training example at each iteration. This makes SGD much faster but also results in noisier updates.

The parameter update rule for SGD is:
$\theta_{new} = \theta_{old} - \alpha \nabla J_i(\theta)$

Where $\nabla J_i(\theta)$ is the gradient computed for the i-th training example.

Let's implement SGD for the same linear regression problem:

In [ ]:
def stochastic_gradient_descent(X, y, learning_rate=0.01, n_iterations=50):
    """
    Implement stochastic gradient descent for linear regression
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples,)
        learning_rate: Learning rate alpha
        n_iterations: Number of iterations (epochs)
        
    Returns:
        theta: Optimized model parameters
        cost_history: Cost at each epoch
    """
    m, n = X.shape
    theta = np.zeros(n + 1)  # Initialize parameters (including bias)
    cost_history = np.zeros(n_iterations)
    
    # Add intercept term
    X_b = np.c_[np.ones((m, 1)), X]
    
    for epoch in range(n_iterations):
        # Shuffle the training data
        indices = np.random.permutation(m)
        X_shuffled = X_b[indices]
        y_shuffled = y[indices]
        
        # Iterate through each training example
        for i in range(m):
            xi = X_shuffled[i:i+1]  # Get single example (maintaining 2D shape)
            yi = y_shuffled[i:i+1]  # Get single target
            
            # Calculate gradient for this example
            prediction = xi.dot(theta)
            error = prediction - yi
            gradient = xi.T.dot(error)
            
            # Update parameters
            theta = theta - learning_rate * gradient
        
        # Calculate cost at end of each epoch (using all data)
        cost = compute_cost(X, y, theta)
        cost_history[epoch] = cost
        
    return theta, cost_history

# Run stochastic gradient descent
learning_rate_sgd = 0.001  # Usually smaller learning rate for SGD
n_iterations_sgd = 50      # Epochs (each epoch processes all examples)
theta_sgd, cost_history_sgd = stochastic_gradient_descent(X, y, learning_rate_sgd, n_iterations_sgd)

# Plot the cost history for SGD
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_iterations_sgd + 1), cost_history_sgd, 'g-')
plt.title('SGD: Cost vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Cost')
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot the regression line from SGD
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='b', label='Data Points')

# Sort X for clean line plotting
X_sorted = np.sort(X, axis=0)
X_b_sorted = np.c_[np.ones((len(X_sorted), 1)), X_sorted]
y_pred_sgd = X_b_sorted.dot(theta_sgd)

plt.plot(X_sorted, y_pred_sgd, 'g-', label=f'SGD: y = {theta_sgd[0]:.2f} + {theta_sgd[1]:.2f}x')
plt.plot(X_sorted, y_pred, 'r--', label=f'Batch GD: y = {theta[0]:.2f} + {theta[1]:.2f}x')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparison: SGD vs Batch Gradient Descent')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Mini-Batch Gradient Descent

Mini-batch gradient descent is a compromise between batch gradient descent and stochastic gradient descent. It updates the parameters using a small random batch of training examples. This approach offers a balance between the efficiency of SGD and the stability of batch gradient descent.

The parameter update rule is:
$\theta_{new} = \theta_{old} - \alpha \nabla J_{mini-batch}(\theta)$

Let's implement mini-batch gradient descent:

In [ ]:
def mini_batch_gradient_descent(X, y, batch_size=10, learning_rate=0.01, n_iterations=100):
    """
    Implement mini-batch gradient descent for linear regression
    
    Args:
        X: Input features (n_samples, n_features)
        y: Target values (n_samples,)
        batch_size: Number of examples in each batch
        learning_rate: Learning rate alpha
        n_iterations: Number of iterations (epochs)
        
    Returns:
        theta: Optimized model parameters
        cost_history: Cost at each epoch
    """
    m, n = X.shape
    theta = np.zeros(n + 1)  # Initialize parameters (including bias)
    cost_history = np.zeros(n_iterations)
    
    # Add intercept term
    X_b = np.c_[np.ones((m, 1)), X]
    
    for epoch in range(n_iterations):
        # Shuffle the training data
        indices = np.random.permutation(m)
        X_shuffled = X_b[indices]
        y_shuffled = y[indices]
        
        # Process mini-batches
        for i in range(0, m, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]
            
            # Calculate gradient for this batch
            predictions = X_batch.dot(theta)
            errors = predictions - y_batch
            gradient = X_batch.T.dot(errors) / len(X_batch)
            
            # Update parameters
            theta = theta - learning_rate * gradient
        
        # Calculate cost at end of each epoch (using all data)
        cost = compute_cost(X, y, theta)
        cost_history[epoch] = cost
        
    return theta, cost_history

# Run mini-batch gradient descent
batch_size = 10
learning_rate_mb = 0.005
n_iterations_mb = 100
theta_mb, cost_history_mb = mini_batch_gradient_descent(X, y, batch_size, learning_rate_mb, n_iterations_mb)

# Plot the cost history for mini-batch GD
plt.figure(figsize=(10, 6))
plt.plot(range(1, n_iterations_mb + 1), cost_history_mb, 'c-')
plt.title(f'Mini-Batch GD (batch_size={batch_size}): Cost vs. Epoch')
plt.xlabel('Epoch')
plt.ylabel('Cost')
plt.grid(True)
plt.tight_layout()
plt.show()

# Compare all three methods
plt.figure(figsize=(12, 8))
plt.scatter(X, y, color='b', alpha=0.3, label='Data Points')

# Plot regression lines
X_sorted = np.sort(X, axis=0)
X_b_sorted = np.c_[np.ones((len(X_sorted), 1)), X_sorted]

y_pred = X_b_sorted.dot(theta)
y_pred_sgd = X_b_sorted.dot(theta_sgd)
y_pred_mb = X_b_sorted.dot(theta_mb)

plt.plot(X_sorted, y_pred, 'r-', linewidth=2, label=f'Batch GD: y = {theta[0]:.2f} + {theta[1]:.2f}x')
plt.plot(X_sorted, y_pred_sgd, 'g-', linewidth=2, label=f'SGD: y = {theta_sgd[0]:.2f} + {theta_sgd[1]:.2f}x')
plt.plot(X_sorted, y_pred_mb, 'c-', linewidth=2, label=f'Mini-Batch GD: y = {theta_mb[0]:.2f} + {theta_mb[1]:.2f}x')

plt.xlabel('X')
plt.ylabel('y')
plt.title('Comparison of Gradient Descent Methods')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Compare the convergence
plt.figure(figsize=(12, 8))

# Normalize iterations to the same scale (0-1)
x_batch = np.linspace(0, 1, len(cost_history))
x_sgd = np.linspace(0, 1, len(cost_history_sgd))
x_mb = np.linspace(0, 1, len(cost_history_mb))

plt.plot(x_batch, cost_history, 'r-', linewidth=2, label='Batch GD')
plt.plot(x_sgd, cost_history_sgd, 'g-', linewidth=2, label='SGD')
plt.plot(x_mb, cost_history_mb, 'c-', linewidth=2, label='Mini-Batch GD')

plt.xlabel('Normalized Training Progress')
plt.ylabel('Cost')
plt.title('Cost Convergence Comparison')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Gradient Descent Variants

Standard gradient descent works well for many problems, but there are several advanced variants that can provide faster convergence and better performance. Let's explore some of these variants:

1. **Momentum**: Helps accelerate convergence and reduce oscillation
2. **RMSprop**: Adapts learning rates based on recent gradients
3. **Adam**: Combines ideas from momentum and RMSprop
4. **AdaGrad**: Adapts learning rates based on historical gradients

Let's implement these variants on our simple 2D function $f(x, y) = x^2 + y^2$.

In [ ]:
# Test function for optimization
def f(x, y):
    return x**2 + y**2

def df(x, y):
    return np.array([2*x, 2*y])

# Implementation of various optimization algorithms
def gradient_descent_variants(variant='standard', learning_rate=0.1, n_iterations=100):
    """
    Implement various gradient descent variants
    
    Args:
        variant: Which variant to use ('standard', 'momentum', 'rmsprop', 'adam')
        learning_rate: Learning rate alpha
        n_iterations: Number of iterations
        
    Returns:
        theta_history: Parameters at each iteration
    """
    # Initialize parameters
    theta = np.array([4.0, 4.0])  # Start away from the minimum
    theta_history = np.zeros((n_iterations + 1, 2))
    theta_history[0] = theta
    
    # Hyperparameters for variants
    beta1 = 0.9  # Momentum coefficient
    beta2 = 0.999  # RMSprop/Adam coefficient
    epsilon = 1e-8  # Small value to prevent division by zero
    
    # Initialize additional variables for variants
    v = np.zeros(2)  # First moment vector (momentum)
    s = np.zeros(2)  # Second moment vector (RMSprop/Adam)
    
    for i in range(1, n_iterations + 1):
        gradient = df(theta[0], theta[1])
        
        if variant == 'standard':
            # Standard gradient descent
            theta = theta - learning_rate * gradient
            
        elif variant == 'momentum':
            # Gradient descent with momentum
            v = beta1 * v - learning_rate * gradient
            theta = theta + v
            
        elif variant == 'rmsprop':
            # RMSprop
            s = beta2 * s + (1 - beta2) * gradient**2
            theta = theta - learning_rate * gradient / (np.sqrt(s) + epsilon)
            
        elif variant == 'adam':
            # Adam optimization
            v = beta1 * v + (1 - beta1) * gradient  # First moment estimate
            s = beta2 * s + (1 - beta2) * gradient**2  # Second moment estimate
            
            # Bias correction
            v_corrected = v / (1 - beta1**(i))
            s_corrected = s / (1 - beta2**(i))
            
            theta = theta - learning_rate * v_corrected / (np.sqrt(s_corrected) + epsilon)
            
        # Store history
        theta_history[i] = theta
        
    return theta_history

# Run optimizations with different variants
variants = ['standard', 'momentum', 'rmsprop', 'adam']
n_iterations = 50
all_trajectories = {}

for variant in variants:
    all_trajectories[variant] = gradient_descent_variants(variant, learning_rate=0.1, n_iterations=n_iterations)

# Plot the trajectories on contour plot
def plot_trajectories():
    # Create a contour plot of the function
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    plt.figure(figsize=(14, 10))
    contour = plt.contour(X, Y, Z, levels=30, cmap=cm.viridis)
    plt.colorbar(contour)
    
    # Plot trajectories for each variant
    colors = {'standard': 'r', 'momentum': 'g', 'rmsprop': 'b', 'adam': 'c'}
    labels = {
        'standard': 'Standard GD', 
        'momentum': 'Momentum', 
        'rmsprop': 'RMSprop', 
        'adam': 'Adam'
    }
    
    for variant, trajectory in all_trajectories.items():
        plt.plot(trajectory[:, 0], trajectory[:, 1], color=colors[variant], 
                 marker='o', markersize=4, linewidth=1.5, 
                 label=labels[variant])
        # Mark start point
        plt.plot(trajectory[0, 0], trajectory[0, 1], 'k*', markersize=10)
    
    # Mark the optimum
    plt.plot(0, 0, 'r*', markersize=10)
    
    plt.title('Comparison of Gradient Descent Variants')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_trajectories()

# Plot the convergence rates
plt.figure(figsize=(12, 8))
for variant, trajectory in all_trajectories.items():
    # Calculate function value at each step
    values = np.array([f(point[0], point[1]) for point in trajectory])
    plt.plot(range(n_iterations + 1), values, label=variant, linewidth=2)

plt.yscale('log')  # Log scale to better see the differences
plt.title('Convergence Comparison of Gradient Descent Variants')
plt.xlabel('Iteration')
plt.ylabel('Function Value (log scale)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Challenges and Solutions

Gradient descent, while powerful, comes with several challenges:

1. **Choosing the right learning rate**: 
   - Too small: slow convergence
   - Too large: overshooting, divergence
   
2. **Local minima and saddle points**:
   - Getting stuck in local minima when the objective function is non-convex
   - Slow progress near saddle points

3. **Plateaus and ravines**:
   - Slow progress in regions where the gradient is close to zero
   - Oscillation in narrow ravines

Let's explore these challenges and their solutions:

In [ ]:
# Let's demonstrate the effect of different learning rates
def learning_rate_experiment(rates=[0.01, 0.1, 0.5, 1.0], n_iterations=50):
    """
    Demonstrate the effect of different learning rates on convergence
    """
    # Function with multiple local minima for demonstration
    def complex_function(x):
        return 0.1 * (x**4 - 16*x**2 + 5*x)
    
    def complex_gradient(x):
        return 0.1 * (4*x**3 - 32*x + 5)
    
    x_values = np.linspace(-5, 5, 1000)
    y_values = complex_function(x_values)
    
    plt.figure(figsize=(15, 10))
    
    # Plot the function
    plt.subplot(2, 1, 1)
    plt.plot(x_values, y_values, 'b-')
    plt.title("Function with Multiple Local Minima")
    plt.xlabel("x")
    plt.ylabel("f(x)")
    plt.grid(True)
    
    # Plot the learning trajectories
    for learning_rate in rates:
        x = 4.0  # Starting point
        x_history = np.zeros(n_iterations + 1)
        x_history[0] = x
        
        for i in range(1, n_iterations + 1):
            gradient = complex_gradient(x)
            x = x - learning_rate * gradient
            x_history[i] = x
        
        plt.plot(x_history, complex_function(x_history), 'o-', label=f'α = {learning_rate}')
    
    plt.legend()
    
    # Plot the convergence
    plt.subplot(2, 1, 2)
    
    for learning_rate in rates:
        x = 4.0  # Starting point
        x_history = np.zeros(n_iterations + 1)
        f_history = np.zeros(n_iterations + 1)
        x_history[0] = x
        f_history[0] = complex_function(x)
        
        for i in range(1, n_iterations + 1):
            gradient = complex_gradient(x)
            x = x - learning_rate * gradient
            x_history[i] = x
            f_history[i] = complex_function(x)
        
        plt.plot(range(n_iterations + 1), f_history, 'o-', label=f'α = {learning_rate}')
    
    plt.title("Convergence with Different Learning Rates")
    plt.xlabel("Iteration")
    plt.ylabel("Function Value")
    plt.grid(True)
    plt.legend()
    
    plt.tight_layout()
    plt.show()

# Run the learning rate experiment
learning_rate_experiment(rates=[0.01, 0.05, 0.1, 0.2])

### Learning Rate Schedules

Learning rate schedules adjust the learning rate during training. Common strategies include:

1. **Time-based decay**: Reduce learning rate after each epoch
2. **Step decay**: Reduce learning rate by a factor after a fixed number of epochs
3. **Exponential decay**: Exponentially decrease learning rate over time

Let's implement a simple time-based decay schedule:

In [ ]:
# Implement gradient descent with learning rate schedule
def gd_with_schedule(schedule_type='constant', initial_lr=0.1, n_iterations=100):
    """
    Implement gradient descent with various learning rate schedules
    
    Args:
        schedule_type: Type of learning rate schedule ('constant', 'time', 'step', 'exponential')
        initial_lr: Initial learning rate
        n_iterations: Number of iterations
        
    Returns:
        theta_history: Parameters at each iteration
        lr_history: Learning rates at each iteration
    """
    # Initialize parameters for our 2D function f(x, y) = x^2 + y^2
    theta = np.array([4.0, 4.0])  # Start away from the minimum
    theta_history = np.zeros((n_iterations + 1, 2))
    theta_history[0] = theta
    
    lr_history = np.zeros(n_iterations + 1)
    lr_history[0] = initial_lr
    
    for i in range(1, n_iterations + 1):
        # Determine learning rate based on schedule
        if schedule_type == 'constant':
            lr = initial_lr
        elif schedule_type == 'time':
            lr = initial_lr / (1 + 0.05 * i)  # Time-based decay
        elif schedule_type == 'step':
            lr = initial_lr * 0.5 ** (i // 10)  # Step decay (halve every 10 iterations)
        elif schedule_type == 'exponential':
            lr = initial_lr * np.exp(-0.1 * i)  # Exponential decay
        else:
            lr = initial_lr  # Default to constant
            
        # Calculate gradient and update parameters
        gradient = df(theta[0], theta[1])
        theta = theta - lr * gradient
        
        # Store history
        theta_history[i] = theta
        lr_history[i] = lr
        
    return theta_history, lr_history

# Run gradient descent with different learning rate schedules
schedules = ['constant', 'time', 'step', 'exponential']
schedule_results = {}
lr_histories = {}

for schedule in schedules:
    theta_history, lr_history = gd_with_schedule(schedule_type=schedule, initial_lr=0.1, n_iterations=50)
    schedule_results[schedule] = theta_history
    lr_histories[schedule] = lr_history

# Plot learning rate schedules
plt.figure(figsize=(15, 6))
plt.subplot(1, 2, 1)

for schedule, lr_history in lr_histories.items():
    plt.plot(range(len(lr_history)), lr_history, label=f'{schedule.capitalize()} Schedule')
    
plt.title('Learning Rate Schedules')
plt.xlabel('Iteration')
plt.ylabel('Learning Rate')
plt.legend()
plt.grid(True)

# Plot convergence with different schedules
plt.subplot(1, 2, 2)

for schedule, theta_history in schedule_results.items():
    # Calculate function value at each step
    values = np.array([f(point[0], point[1]) for point in theta_history])
    plt.semilogy(range(len(values)), values, label=f'{schedule.capitalize()} Schedule')

plt.title('Convergence with Different Learning Rate Schedules')
plt.xlabel('Iteration')
plt.ylabel('Function Value (log scale)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Plot the trajectories with different schedules
def plot_schedule_trajectories():
    # Create a contour plot of the function
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    plt.figure(figsize=(12, 10))
    contour = plt.contour(X, Y, Z, levels=20, cmap=cm.viridis)
    plt.colorbar(contour)
    
    # Plot trajectories for each schedule
    colors = {'constant': 'r', 'time': 'g', 'step': 'b', 'exponential': 'c'}
    
    for schedule, trajectory in schedule_results.items():
        plt.plot(trajectory[:, 0], trajectory[:, 1], color=colors[schedule], 
                 marker='o', markersize=4, linewidth=1.5, 
                 label=f'{schedule.capitalize()} Schedule')
        # Mark start point
        plt.plot(trajectory[0, 0], trajectory[0, 1], 'k*', markersize=10)
    
    # Mark the optimum
    plt.plot(0, 0, 'r*', markersize=10)
    
    plt.title('Optimization Trajectories with Different Learning Rate Schedules')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

plot_schedule_trajectories()

## Visualization of Convergence

Let's create more advanced visualizations to better understand the convergence process of gradient descent. We'll create:

1. A 3D visualization of the optimization landscape
2. An animation of the optimization process

In [ ]:
# Create a function to visualize the optimization landscape in 3D
def visualize_3d_optimization(theta_history):
    # Create 3D figure
    fig = plt.figure(figsize=(12, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Create surface for our function f(x, y) = x^2 + y^2
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    # Plot the surface
    surface = ax.plot_surface(X, Y, Z, cmap=cm.coolwarm, alpha=0.7)
    
    # Plot the optimization trajectory
    trajectory_x = theta_history[:, 0]
    trajectory_y = theta_history[:, 1]
    trajectory_z = np.array([f(x, y) for x, y in theta_history])
    
    ax.plot(trajectory_x, trajectory_y, trajectory_z, 'r-', linewidth=2, marker='o', markersize=4)
    
    # Mark the starting point
    ax.scatter(trajectory_x[0], trajectory_y[0], trajectory_z[0], color='green', s=100, label='Start')
    
    # Mark the minimum point
    ax.scatter(0, 0, 0, color='black', s=100, label='Minimum')
    
    # Set labels and title
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('f(X, Y)')
    ax.set_title('3D Visualization of Gradient Descent')
    ax.legend()
    
    plt.tight_layout()
    plt.show()

# Visualize the optimization process in 3D for standard gradient descent
visualize_3d_optimization(all_trajectories['standard'])

In [ ]:
# Create an animation of the contour plot with the optimization trajectory
def create_animation():
    # Setup the figure and axis
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create contour plot
    x = np.linspace(-5, 5, 100)
    y = np.linspace(-5, 5, 100)
    X, Y = np.meshgrid(x, y)
    Z = f(X, Y)
    
    contour = ax.contour(X, Y, Z, levels=30, cmap=cm.viridis)
    plt.colorbar(contour, ax=ax)
    
    # Get trajectory for animation (using Adam for smooth convergence)
    trajectory = all_trajectories['adam']
    
    # Initialize a line with the starting point
    line, = ax.plot([], [], 'r-', linewidth=2, marker='o', markersize=6)
    point, = ax.plot([], [], 'ro', markersize=10)
    
    # Add title and labels
    ax.set_title('Gradient Descent Optimization (Adam)')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.grid(True)
    
    # Mark the optimal point
    ax.plot(0, 0, 'k*', markersize=15)
    
    # Set limits
    ax.set_xlim(-5, 5)
    ax.set_ylim(-5, 5)
    
    # Initialization function for animation
    def init():
        line.set_data([], [])
        point.set_data([], [])
        return line, point
    
    # Animation function
    def animate(i):
        # Update the line with all points up to the current iteration
        line.set_data(trajectory[:i+1, 0], trajectory[:i+1, 1])
        # Update the current point
        point.set_data(trajectory[i, 0], trajectory[i, 1])
        return line, point
    
    # Create the animation
    anim = animation.FuncAnimation(
        fig, animate, init_func=init,
        frames=len(trajectory), interval=200, blit=True)
    
    # Close the figure to prevent display in notebook
    plt.close(fig)
    
    return HTML(anim.to_jshtml())

# Create and display the animation
animation_html = create_animation()
animation_html

## Practical Examples

Now let's apply gradient descent to some practical machine learning problems:

1. Linear Regression
2. Logistic Regression

We'll implement both from scratch using gradient descent.

In [ ]:
# Generate a more complex regression dataset
np.random.seed(42)
X_multi = np.random.randn(200, 3)  # 3 features
true_theta = np.array([2.5, -1.0, 3.0, -0.5])  # Including bias term
X_b_multi = np.c_[np.ones((200, 1)), X_multi]  # Add intercept term
y_multi = X_b_multi.dot(true_theta) + np.random.randn(200) * 0.5  # Add noise

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(X_multi, y_multi, test_size=0.3, random_state=42)

# Linear Regression using batch gradient descent
def linear_regression_gd(X, y, learning_rate=0.01, n_iterations=1000):
    """
    Implement linear regression using batch gradient descent
    """
    m, n = X.shape
    theta = np.zeros(n + 1)  # Initialize parameters (including bias)
    X_b = np.c_[np.ones((m, 1)), X]  # Add intercept term
    
    # Store cost history for plotting
    cost_history = np.zeros(n_iterations)
    
    for i in range(n_iterations):
        # Compute predictions
        predictions = X_b.dot(theta)
        
        # Compute errors
        errors = predictions - y
        
        # Compute gradients
        gradients = X_b.T.dot(errors) / m
        
        # Update parameters
        theta = theta - learning_rate * gradients
        
        # Compute cost
        cost = np.mean(errors**2) / 2
        cost_history[i] = cost
    
    return theta, cost_history

# Train the model using gradient descent
theta_lr, cost_history_lr = linear_regression_gd(X_train, y_train, learning_rate=0.01, n_iterations=1000)

# Evaluate the model
def evaluate_regression(X, y, theta):
    X_b = np.c_[np.ones((len(X), 1)), X]
    y_pred = X_b.dot(theta)
    mse = mean_squared_error(y, y_pred)
    return mse, y_pred

# Calculate MSE for train and test sets
train_mse, y_train_pred = evaluate_regression(X_train, y_train, theta_lr)
test_mse, y_test_pred = evaluate_regression(X_test, y_test, theta_lr)

print(f"Linear Regression using Gradient Descent:")
print(f"True parameters: {true_theta}")
print(f"Estimated parameters: {theta_lr}")
print(f"Training MSE: {train_mse:.4f}")
print(f"Test MSE: {test_mse:.4f}")

# Plot the cost history
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cost_history_lr) + 1), cost_history_lr, 'b-')
plt.title('Cost vs. Iteration for Linear Regression')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.grid(True)
plt.tight_layout()
plt.show()

# Compare predictions with actual values
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.7)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--')
plt.title('Training Data: Actual vs Predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.7)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('Test Data: Actual vs Predicted')
plt.xlabel('Actual')
plt.ylabel('Predicted')
plt.grid(True)

plt.tight_layout()
plt.show()

### Logistic Regression from Scratch

Now let's implement logistic regression using gradient descent for a binary classification problem.

In [ ]:
# Generate a classification dataset
np.random.seed(42)
X_class, y_class = make_classification(
    n_samples=200, 
    n_features=2, 
    n_informative=2, 
    n_redundant=0, 
    n_clusters_per_class=1, 
    random_state=42
)

# Split into train/test sets
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_class, y_class, test_size=0.3, random_state=42)

# Plot the dataset
plt.figure(figsize=(10, 6))
plt.scatter(X_class[y_class==0, 0], X_class[y_class==0, 1], label='Class 0', alpha=0.7)
plt.scatter(X_class[y_class==1, 0], X_class[y_class==1, 1], label='Class 1', alpha=0.7)
plt.title('Classification Dataset')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Logistic regression using gradient descent
def logistic_regression_gd(X, y, learning_rate=0.1, n_iterations=1000):
    """
    Implement logistic regression using batch gradient descent
    """
    m, n = X.shape
    theta = np.zeros(n + 1)  # Initialize parameters (including bias)
    X_b = np.c_[np.ones((m, 1)), X]  # Add intercept term
    
    # Store cost history for plotting
    cost_history = np.zeros(n_iterations)
    
    for i in range(n_iterations):
        # Calculate the hypothesis (predicted probabilities)
        z = X_b.dot(theta)
        h = sigmoid(z)
        
        # Calculate the cost (negative log-likelihood)
        epsilon = 1e-10  # Small value to avoid log(0)
        cost = -np.mean(y * np.log(h + epsilon) + (1 - y) * np.log(1 - h + epsilon))
        cost_history[i] = cost
        
        # Calculate the gradient
        gradient = X_b.T.dot(h - y) / m
        
        # Update parameters
        theta = theta - learning_rate * gradient
    
    return theta, cost_history

# Train logistic regression model using gradient descent
theta_log, cost_history_log = logistic_regression_gd(X_train_c, y_train_c, learning_rate=0.1, n_iterations=1000)

# Function to predict class probabilities and labels
def predict_proba(X, theta):
    X_b = np.c_[np.ones((len(X), 1)), X]
    return sigmoid(X_b.dot(theta))

def predict(X, theta, threshold=0.5):
    return predict_proba(X, theta) >= threshold

# Calculate accuracy
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

# Make predictions
y_train_pred_c = predict(X_train_c, theta_log)
y_test_pred_c = predict(X_test_c, theta_log)

# Calculate accuracy
train_accuracy = accuracy(y_train_c, y_train_pred_c)
test_accuracy = accuracy(y_test_c, y_test_pred_c)

print(f"Logistic Regression using Gradient Descent:")
print(f"Estimated parameters: {theta_log}")
print(f"Training accuracy: {train_accuracy:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

# Plot the cost history
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(cost_history_log) + 1), cost_history_log, 'b-')
plt.title('Cost vs. Iteration for Logistic Regression')
plt.xlabel('Iteration')
plt.ylabel('Cost')
plt.grid(True)
plt.tight_layout()
plt.show()

# Visualize the decision boundary
def plot_decision_boundary(X, y, theta):
    # Set min and max values for plotting
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    
    # Create a grid of points
    xx1, xx2 = np.meshgrid(np.linspace(x1_min, x1_max, 100),
                          np.linspace(x2_min, x2_max, 100))
    
    # Make predictions for each point in the grid
    grid = np.c_[xx1.ravel(), xx2.ravel()]
    probs = predict_proba(grid, theta)
    probs = probs.reshape(xx1.shape)
    
    # Plot the decision boundary and data points
    plt.figure(figsize=(10, 8))
    plt.contourf(xx1, xx2, probs, alpha=0.3, cmap=plt.cm.Paired)
    plt.contour(xx1, xx2, probs, levels=[0.5], linewidths=2, colors='white')
    
    # Plot the data points
    plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', label='Class 0', alpha=0.7)
    plt.scatter(X[y==1, 0], X[y==1, 1], c='red', label='Class 1', alpha=0.7)
    
    plt.title('Logistic Regression Decision Boundary')
    plt.xlabel('Feature 1')
    plt.ylabel('Feature 2')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Plot the decision boundary for the training data
plot_decision_boundary(X_train_c, y_train_c, theta_log)

# Calculate and visualize metrics
from sklearn.metrics import confusion_matrix, classification_report

# Calculate confusion matrix
cm_test = confusion_matrix(y_test_c, y_test_pred_c)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
plt.imshow(cm_test, cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
class_names = ['Class 0', 'Class 1']
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names)
plt.yticks(tick_marks, class_names)

# Add text annotations
threshold = cm_test.max() / 2
for i in range(2):
    for j in range(2):
        plt.text(j, i, f'{cm_test[i, j]}',
                 horizontalalignment='center',
                 color='white' if cm_test[i, j] > threshold else 'black')

plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()

# Print classification report
print("Classification Report:")
print(classification_report(y_test_c, y_test_pred_c, target_names=class_names))

## Summary

In this notebook, we explored gradient descent optimization, a fundamental algorithm in machine learning and deep learning. We covered:

1. **Basic Concepts**: Understanding how gradient descent works by iteratively updating parameters in the direction of steepest descent.

2. **Mathematical Foundations**: The mathematics behind gradients and update rules.

3. **Gradient Descent Variants**:
   - Batch Gradient Descent
   - Stochastic Gradient Descent (SGD)
   - Mini-Batch Gradient Descent

4. **Advanced Optimization Algorithms**:
   - Momentum
   - RMSprop
   - Adam

5. **Learning Rate Considerations**:
   - Effects of different learning rates
   - Learning rate schedules

6. **Visualization Techniques**:
   - 2D and 3D visualization of optimization trajectories
   - Animated visualization of the optimization process

7. **Practical Applications**:
   - Linear regression from scratch
   - Logistic regression from scratch

Gradient descent is at the heart of most deep learning algorithms, and understanding its principles and challenges is essential for building and optimizing machine learning models.

## References

1. Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.
2. Ruder, S. (2016). An overview of gradient descent optimization algorithms. arXiv preprint arXiv:1609.04747.
3. Kingma, D. P., & Ba, J. (2014). Adam: A method for stochastic optimization. arXiv preprint arXiv:1412.6980.
4. Bottou, L. (2010). Large-scale machine learning with stochastic gradient descent. In Proceedings of COMPSTAT'2010 (pp. 177-186). Physica-Verlag HD.
5. Nesterov, Y. (1983). A method for unconstrained convex minimization problem with the rate of convergence O(1/k^2). In Doklady Akademii Nauk (Vol. 269, pp. 543-547).